__????? ?.?.__
__v.01__
__23/12/2022__

__???????__

1. ????????? ???????, ????? ? ???????. 
2. ??? ?????????? ?????????? ?? ???????????? ??????? ???????. 
3. ?? ????????? ????? CatBoostClassifier ??? ??????? ??????? ????????. 
4. ???????? ????????? ? ??????????? ????? ?????????. 
5. ??? ????????????? ? ?????????????????? ??????????? ???????????? ??????????? ????? ??????. 
6. ??? ?????????? ???????? - ?????? ?????????? ????? ?????? ?? ??????? ?????? ?????????? ?????? (????????? ????????????? ??????). ?????????? ?????? ??????? % ?? ?????? ?? ??????? ???????? ??????? ??????. ???????? ??????? 20 ???????? ?????? ?? ?????? ????? ?? 5% ?? ????? ?????????????.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from math import sqrt
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC


from catboost import CatBoostClassifier
from sklearn.svm import SVC

from sklearn.model_selection import validation_curve, learning_curve

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df=pd.read_csv('winequality-white.csv',
              sep=';')

In [ ]:
df

In [ ]:
df.loc[df.quality<=5, 'quality']=0
df.loc[df.quality>=6, 'quality']=1

In [ ]:
df.quality.value_counts()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop(df.loc[df.duplicated()].index, axis=0, inplace=True)

In [ ]:
X=df.drop('quality', axis=1)
y=df.quality

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [ ]:
results = {}

## I. ?????? ??????????? ???????? ??? ?????? - Catboost classifier

In [ ]:
catb_pipeline = Pipeline([('scale', StandardScaler()),
                          ('catb', CatBoostClassifier(logging_level='Silent'))    
                        ])
catb_pipeline.fit(X_train, y_train)

In [ ]:
y_pred=catb_pipeline.predict(X_test)

In [ ]:
roc_auc_score(y_test, y_pred)

In [ ]:
results ['CatBoost'] = roc_auc_score(y_test, y_pred)

## II. ????????

#### 01. Voting Classifier ?? ???????

In [ ]:
from sklearn.ensemble import VotingClassifier

dtc = DecisionTreeClassifier()
svc = SVC (probability=True)
knn = KNeighborsClassifier()


voting_clf = VotingClassifier(
    estimators=[('svc', svc), ('dtc', dtc), ('knn', knn)],
    voting='hard')

voting_pipe = Pipeline([('scale', StandardScaler()),
                          ('voting', voting_clf)    
                        ])

In [ ]:
voting_pipe.fit(X_train, y_train)

In [ ]:
y_pred=voting_pipe.predict(X_test)
roc_auc_score(y_test, y_pred)

In [ ]:
results ['Voting (BOX)'] = roc_auc_score(y_test, y_pred)

#### 02. Voting Classifier - ??????????? ??????????. 
#### HARD 

In [ ]:
#??????? DecisionTreeClassifier ? ?????? ???????.
dtc = DecisionTreeClassifier()
dtc.fit (StandardScaler().fit_transform(X_train), y_train)
predict_dtc = dtc.predict (StandardScaler().fit_transform(X_test))

#??????? SVC ? ?????? ???????.
svc = SVC(probability=True)
svc.fit (StandardScaler().fit_transform(X_train), y_train)
predict_svc = svc.predict (StandardScaler().fit_transform(X_test))

#??????? KNeighborsClassifier ? ?????? ???????.
knn = KNeighborsClassifier()
knn.fit (StandardScaler().fit_transform(X_train), y_train)
predict_knn = knn.predict (StandardScaler().fit_transform(X_test))

In [ ]:
hard = predict_dtc + predict_svc + predict_knn

In [ ]:
roc_auc_score(y_test, (hard>1).astype(int))

In [ ]:
results ['Voting (OWN_HARD)'] = roc_auc_score(y_test, (hard>1).astype(int))

#### SOFT

In [ ]:
#??????? DecisionTreeClassifier ? ?????? ???????.
dtc = DecisionTreeClassifier()
dtc.fit (StandardScaler().fit_transform(X_train), y_train)
predict_dtc = dtc.predict_proba (StandardScaler().fit_transform(X_test))

#??????? LogisticRegression ? ?????? ???????.
svc = SVC(probability=True)
svc.fit (StandardScaler().fit_transform(X_train), y_train)
predict_svc = svc.predict_proba (StandardScaler().fit_transform(X_test))

#??????? KNeighborsClassifier ? ?????? ???????.
knn = KNeighborsClassifier()
knn.fit (StandardScaler().fit_transform(X_train), y_train)
predict_knn = knn.predict_proba (StandardScaler().fit_transform(X_test))

In [ ]:
soft = predict_dtc + predict_svc + predict_knn

In [ ]:
roc_auc_score(y_test, np.argmax(soft, axis=1))

In [ ]:
results ['Voting (OWN_SOFT)'] = roc_auc_score(y_test, np.argmax(soft, axis=1))

#### 03. Stacking classifier - ?? ???????

In [ ]:
from sklearn.ensemble import StackingClassifier

dtc = DecisionTreeClassifier()
lr = LogisticRegression(max_iter=1000)
knn = KNeighborsClassifier()

stacking_clf = StackingClassifier(
    estimators=[('dtc', dtc), ('lr', lr), ('knn', knn)],
    final_estimator=lr, 
    n_jobs=-1, 
    verbose=1)

stacking_pipe = Pipeline([('scale', StandardScaler()),
                          ('stacking', stacking_clf)    
                        ])

In [ ]:
stacking_pipe.fit(X_train, y_train)

In [ ]:
y_pred=stacking_pipe.predict(X_test)

In [ ]:
roc_auc_score(y_test, y_pred)

In [ ]:
results ['Stacking (BOX)'] = roc_auc_score(y_test, y_pred)

#### 04. Stacking classifier - ??????????? ??????????

In [ ]:
dtc = DecisionTreeClassifier()
dtc.fit (StandardScaler().fit_transform(X_train), y_train)
predict_dtc = dtc.predict (StandardScaler().fit_transform(X_test))

lr = LogisticRegression(max_iter=1000)
lr.fit (StandardScaler().fit_transform(X_train), y_train)
predict_lr = lr.predict (StandardScaler().fit_transform(X_test))

knn = KNeighborsClassifier()
knn.fit (StandardScaler().fit_transform(X_train), y_train)
predict_knn = knn.predict (StandardScaler().fit_transform(X_test))

In [ ]:
predict_stack = np.stack ((predict_dtc, predict_lr, predict_knn), axis = 1)
lr. fit (predict_stack, y_test)
y_pred = lr.predict(predict_stack)
roc_auc_score(y_test, y_pred)

In [ ]:
results ['Stacking (OWN)'] = roc_auc_score(y_test, y_pred)

#### 05. Bagging classifier - ?? ???????

In [ ]:
from sklearn.ensemble import BaggingClassifier

bagging_clf = BaggingClassifier(lr, n_estimators=1000,
    max_samples=100, bootstrap=True, n_jobs=-1, random_state=42)

bagging_pipe = Pipeline([('scale', StandardScaler()),
                          ('bagging', bagging_clf)    
                        ])

In [ ]:
bagging_pipe.fit(X_train, y_train)

In [ ]:
y_pred=bagging_pipe.predict(X_test)
roc_auc_score(y_test, y_pred)

In [ ]:
results ['Bagging (BOX)'] = roc_auc_score(y_test, y_pred)

#### 06. Bagging classifier - ??????????? ??????????

In [ ]:
class SimpleBagging:
    '''
    model - ML model
    n_iter - total number of iterations
    sample_size - (0-1] - percent of data set, that takes a part in fitting process
    '''
    def __init__ (self, model, n_iter, sample_size):
        self.model = model
        self.n_iter = n_iter
        self.sample_size = sample_size
        self.models = [] ### ?????? ? ?????????? ????????
        self.score = []  ### ?????? ? ???????????? ????????????
    
    def fit (self, X_train, y_train):
        
        for i in range (self.n_iter):
            
            self.model ()        
            
            rng = np.random.default_rng()
            
            rnd = rng.integers(0, len(X_train), 
                               size=int(self.sample_size*len(X_train)), 
                               dtype=np.int64)
    
            X_train_short = X_train.iloc[rnd]
            y_train_short = y_train.iloc[rnd]

            self.models.append (self.model().fit (X_train_short, y_train_short))
    
    ### ? ???????? ???????? ???? HARD ???????????. ?? ??????? ???????? ????????? ???? ???????? ??????? ????? ???????.
    def predict (self, X_test):
        for model in self.models:
            self.score.append (model.predict (X_test))
        hard = np.sum (np.array (self.score).T, axis=1)
        return (hard>(int(self.n_iter/2)+1)).astype(int)

In [ ]:
b = SimpleBagging (DecisionTreeClassifier, 100, 0.4)

In [ ]:
b.fit (X_train, y_train)

In [ ]:
bagging_predict = roc_auc_score (y_test, b.predict(X_test))
bagging_predict

In [ ]:
results ['Bagging (OWN)'] = bagging_predict

##### 07. Boosting classifier - ?? ???????

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

clf = AdaBoostClassifier (n_estimators=100, random_state=0)

clf.fit (X_train, y_train)
roc_auc_score(y_test, clf.predict(X_test))

In [ ]:
results ['Boosting (BOX)'] = roc_auc_score(y_test, clf.predict(X_test))

##### 08. Boosting classifier - ??????????? ??????????

In [ ]:
class SimpleBoost ():
    
    '''
    SimpleBoost - is a class, demonstrates how boosting works. It has 3 parametrs:
    1. model - is a ML model
    2. n_drop - [0-1] is a pecent of correctly predicted y_values to remove on each itearation
    3. n_iter - [0-inf) is a number of iteration
    '''
    
    def __init__ (self, model, n_drop=0.05, n_iter=5):
        self.model = model
        self.n_drop=n_drop
        self.n_iter=n_iter
    
    def fit (self, X_train, y_train):
        
        for x in range (self.n_iter):
            
            model = self.model.fit (X_train, y_train)
            y_pred = model.predict(X_train)
            
            ind_to_drop = int(sum(y_pred==y_train)*self.n_drop)
            
            X_train = X_train.drop(X_train.loc[y_pred==y_train].index[:ind_to_drop], 
                                   axis=0)
            y_train = y_train.drop(y_train.loc[y_pred==y_train].index[:ind_to_drop], 
                                   axis=0)
        
        self.model = model
        print(model)
        
    def predict (self, X_test):
        return (self.model.predict (X_test))       

In [ ]:
a = SimpleBoost (RandomForestClassifier(), 
                 n_drop=0.05, 
                 n_iter=15)
a.fit (X_train, y_train)

In [ ]:
roc_auc_score(y_test, a.predict(X_test))

In [ ]:
results ['Boosting (OWN)'] = roc_auc_score(y_test, a.predict(X_test))

#### ??????????

In [ ]:
plt.figure (figsize=(20,10))
sns.histplot (data=results,
             x=results.keys(),
             y=results.values());
plt.title ('Results of different ML models')

In [ ]:
results

### ????????? ?????????? ?????????????????? CatBoost - 0.7262, Stacking (BOX) - 0.7228, Bagging (OWN) - 0.7332, Boosting (OWN) - 0.732